
# Week 10 Solution Key: Seasonality, SARIMA, and SARIMAX

This instructor notebook follows the Week 10 flow:
1. Understand seasonality in Tetuan city electricity demand.
2. Use seasonal differencing to stationarize the series.
3. Build a SARIMA baseline.
4. Upgrade to SARIMAX using weather exogenous variables.
5. Compare model quality (AIC/BIC) and forecast accuracy.


In [ ]:

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')


## 1) Data loading and hourly preprocessing

In [ ]:

def discover_hourly_path():
    candidates = [
        Path('data/hourly_tetuan_power.csv'),
        Path('/kaggle/working/data/hourly_tetuan_power.csv'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

hourly_path = discover_hourly_path()
if hourly_path is None:
    raise FileNotFoundError(
        "Hourly dataset not found. Run `python generate_dataset.py` first "
        "(or create data/hourly_tetuan_power.csv)."
    )

df = pd.read_csv(hourly_path, parse_dates=['DateTime'])
df = df.set_index('DateTime').sort_index()

print('Loaded:', hourly_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()


## 2) Target selection and seasonality overview

In [ ]:

TARGET = 'Zone 1 Power Consumption'
EXOG_COLS = ['Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows']

missing_exog = [c for c in EXOG_COLS if c not in df.columns]
if missing_exog:
    raise ValueError(f'Missing exogenous columns: {missing_exog}')

y = df[TARGET].asfreq('h')
X = df[EXOG_COLS].asfreq('h')

ax = y.plot(figsize=(14, 4), title='Hourly Zone 1 Power Consumption')
ax.set_ylabel('Power consumption')
plt.show()


In [ ]:

# Daily seasonal structure (24-hour pattern)
hourly_profile = y.groupby(y.index.hour).mean()
ax = hourly_profile.plot(marker='o', figsize=(10, 4), title='Average Consumption by Hour of Day')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Mean consumption')
plt.show()


## 3) Decomposition and seasonal differencing

In [ ]:

decomp = seasonal_decompose(y, model='additive', period=24)
fig = decomp.plot()
fig.set_size_inches(14, 9)
plt.show()


In [ ]:

def adf_report(series, name):
    result = adfuller(series.dropna())
    return pd.Series({
        'series': name,
        'adf_stat': result[0],
        'p_value': result[1],
        'n_obs': result[3],
        'is_stationary_5pct': result[1] < 0.05,
    })

adf_raw = adf_report(y, 'raw')
y_seasonal_diff = y.diff(24)
adf_seasonal_diff = adf_report(y_seasonal_diff, 'seasonal_diff_lag24')

pd.DataFrame([adf_raw, adf_seasonal_diff])


## 4) ACF/PACF-based identification (after seasonal differencing)

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(y_seasonal_diff.dropna(), lags=72, ax=axes[0])
plot_pacf(y_seasonal_diff.dropna(), lags=72, ax=axes[1], method='ywm')
axes[0].set_title('ACF of seasonally differenced series')
axes[1].set_title('PACF of seasonally differenced series')
plt.tight_layout()
plt.show()



## 5) Train-test split

We hold out the last 7 days (`24 * 7 = 168` hourly points) for forecast evaluation.


In [ ]:

horizon = 24 * 7

y_train, y_test = y.iloc[:-horizon], y.iloc[-horizon:]
X_train, X_test = X.iloc[:-horizon], X.iloc[-horizon:]

print('Train points:', len(y_train))
print('Test points :', len(y_test))
print('Train range :', y_train.index.min(), '->', y_train.index.max())
print('Test range  :', y_test.index.min(), '->', y_test.index.max())


## 6) Fit SARIMA baseline

In [ ]:

# Candidate order informed by ACF/PACF and daily seasonality
order = (1, 0, 1)
seasonal_order = (1, 1, 1, 24)

sarima = SARIMAX(
    y_train,
    order=order,
    seasonal_order=seasonal_order,
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_res = sarima.fit(disp=False)
print(sarima_res.summary())


## 7) Fit SARIMAX with weather exogenous variables

In [ ]:

sarimax = SARIMAX(
    y_train,
    exog=X_train,
    order=order,
    seasonal_order=seasonal_order,
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarimax_res = sarimax.fit(disp=False)
print(sarimax_res.summary())


## 8) Forecast and compare models (AIC/BIC + forecast metrics)

In [ ]:

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

sarima_fc = sarima_res.get_forecast(steps=horizon).predicted_mean
sarimax_fc = sarimax_res.get_forecast(steps=horizon, exog=X_test).predicted_mean

comparison = pd.DataFrame([
    {
        'model': 'SARIMA',
        'AIC': sarima_res.aic,
        'BIC': sarima_res.bic,
        'MAE': mean_absolute_error(y_test, sarima_fc),
        'RMSE': rmse(y_test, sarima_fc),
    },
    {
        'model': 'SARIMAX',
        'AIC': sarimax_res.aic,
        'BIC': sarimax_res.bic,
        'MAE': mean_absolute_error(y_test, sarimax_fc),
        'RMSE': rmse(y_test, sarimax_fc),
    },
]).set_index('model').sort_values('RMSE')

comparison


In [ ]:

plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test, label='Actual', color='black', linewidth=2)
plt.plot(y_test.index, sarima_fc, label='SARIMA forecast', alpha=0.8)
plt.plot(y_test.index, sarimax_fc, label='SARIMAX forecast', alpha=0.8)
plt.title('Hold-out Forecast Comparison (Last 7 Days)')
plt.xlabel('Datetime')
plt.ylabel('Power consumption')
plt.legend()
plt.tight_layout()
plt.show()


## 9) Residual diagnostics

In [ ]:

sarimax_res.plot_diagnostics(figsize=(14, 10))
plt.tight_layout()
plt.show()


In [ ]:

lb = acorr_ljungbox(sarimax_res.resid.dropna(), lags=[24, 48], return_df=True)
lb


In [ ]:

# Exogenous variable significance
exog_pvalues = sarimax_res.pvalues[[c for c in sarimax_res.pvalues.index if c in EXOG_COLS]]
exog_pvalues.sort_values()



## 10) Interpretation checklist

- **Seasonality**: Electricity demand has a clear daily cycle (24-hour seasonality).
- **Seasonal differencing**: `D=1, s=24` is typically sufficient to remove repeating daily structure.
- **Modeling**: SARIMA captures autoregressive + seasonal memory; SARIMAX adds weather drivers.
- **Practical value**: Better forecast accuracy supports grid dispatch planning, demand-response scheduling, and operational cost control.
